<small>## 01 — Crawling forum Anime/Game
Mengumpulkan thread publik dari sumber yang dipilih. Gunakan jeda permintaan dan hentikan bila akses dibatasi. Parser tiap situs mungkin perlu penyesuaian berdasarkan HTML aktual.</small>

In [1]:
%pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests, time, json, re, hashlib
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pathlib import Path
from datetime import datetime

ROOT = Path(r"D:\\semester 5\\penelusuran informasi\\searchengine")
RAW = ROOT / "data" / "raw"
AUDIT = ROOT / "data" / "audit"
RAW.mkdir(parents=True, exist_ok=True)
AUDIT.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Student research crawler for public forum content (contact: research project)"}
DELAY = 2.0


In [3]:
FORUMS = {
    "MAL": "https://myanimelist.net/forum/?board=1",
    "Steam": "https://steamcommunity.com/app/391220/discussions/",
    "AniList": "https://anilist.co/forum/recent?category=1",
    "PCGamer": "https://forums.pcgamer.com/forums/general-gaming.33/",
    "AnimeForums": "https://animeforums.net/forum/8-anime-discussion/",
    "AnimeSuki": "https://forums.animesuki.com/forumdisplay.php?f=33",
}
API_SITES = ["anime", "gaming"]


In [4]:
def get_soup(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code in (403, 429):
            print("Access limited:", r.status_code, url)
            return None, r.status_code
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser"), r.status_code
    except Exception as e:
        print("Request failed:", url, str(e)[:200])
        return None, None

def text_clean(s):
    return re.sub(r"\\s+", " ", s or "").strip()

def test_sources():
    rows=[]
    for source, url in FORUMS.items():
        soup, status = get_soup(url)
        rows.append({"source":source, "url":url, "status":status,
                     "accessible":soup is not None,
                     "title": soup.title.get_text(" ",strip=True) if soup and soup.title else ""})
        time.sleep(DELAY)
    return rows

source_status = test_sources()
source_status


[{'source': 'MAL',
  'url': 'https://myanimelist.net/forum/?board=1',
  'status': 200,
  'accessible': True,
  'title': 'Anime Discussion - Forums - MyAnimeList.net'},
 {'source': 'Steam',
  'url': 'https://steamcommunity.com/app/391220/discussions/',
  'status': 200,
  'accessible': True,
  'title': 'Rise of the Tomb Raider General Discussions :: Steam Community'},
 {'source': 'AniList',
  'url': 'https://anilist.co/forum/recent?category=1',
  'status': 200,
  'accessible': True,
  'title': 'Forum - Anime & Manga Discussion · AniList'},
 {'source': 'PCGamer',
  'url': 'https://forums.pcgamer.com/forums/general-gaming.33/',
  'status': 200,
  'accessible': True,
  'title': 'Checking your browser'},
 {'source': 'AnimeForums',
  'url': 'https://animeforums.net/forum/8-anime-discussion/',
  'status': 200,
  'accessible': True,
  'title': 'Anime Discussion - Anime Forums'},
 {'source': 'AnimeSuki',
  'url': 'https://forums.animesuki.com/forumdisplay.php?f=33',
  'status': 200,
  'accessibl

In [5]:
# Thread-link discovery for each forum board (heuristic; inspect links if selectors need adjustment)
def discover_thread_links(source, board_url, max_board_pages=10):
    links=set()
    pages=[board_url]
    visited=set()
    for page_url in pages[:max_board_pages]:
        if page_url in visited: continue
        visited.add(page_url)
        soup, status=get_soup(page_url)
        if not soup: continue
        for a in soup.find_all("a", href=True):
            href=urljoin(page_url, a["href"])
            text=text_clean(a.get_text(" "))
            if not text or len(text)<4: continue
            p=urlparse(href)
            if source=="MAL" and "/forum/" in p.path and ("topicid=" in href or "topic/" in p.path):
                links.add(href.split("#")[0])
            elif source=="Steam" and "/discussions/" in p.path and "announcements" not in p.path and p.path.count("/")>=4:
                links.add(href.split("#")[0])
            elif source=="AniList" and "/forum/" in p.path and "recent" not in p.path:
                links.add(href.split("#")[0])
            elif source=="PCGamer" and ("threads/" in p.path):
                links.add(href.split("#")[0])
            elif source=="AnimeForums" and ("threads/" in p.path):
                links.add(href.split("#")[0])
            elif source=="AnimeSuki" and ("showthread.php" in p.path or "topic" in p.path):
                links.add(href.split("#")[0])
        time.sleep(DELAY)
    return sorted(links)



In [ ]:
# Ekstrak hread title dan visible post text yang terlihat menggunakan selektor umum; selektor khusus situs dapat disempurnakan.
def extract_thread(source, url):
    soup, status=get_soup(url)
    if not soup: return None
    title=(soup.title.get_text(" ",strip=True) if soup.title else "")
    title=text_clean(title)
    selectors={
        "MAL":[".message", ".forum-topic-message", ".topic-message"],
        "Steam":[".forum_topic", ".commentthread_text", ".discussion"],
        "AniList":[".markdown", ".forum-thread", ".activity-entry"],
        "PCGamer":[".message-body", ".message-content", ".bbWrapper"],
        "AnimeForums":[".message-body", ".message-content", ".bbWrapper"],
        "AnimeSuki":[".postbody", ".post_message", ".content"],
    }
    chunks=[]
    for sel in selectors.get(source,[]):
        found=soup.select(sel)
        if found:
            chunks=[text_clean(x.get_text(" ")) for x in found]
            chunks=[x for x in chunks if len(x)>30]
            if chunks: break
    if not chunks:
        chunks=[text_clean(soup.get_text(" "))[:12000]]
    content="\\n".join(chunks)
    if len(content)<50: return None
    return {"source":source,"title":title,"content":content,"url":url,
            "crawl_time":datetime.utcnow().isoformat()+"Z"}



In [ ]:
# Lakukan crawling pada semua utas forum yang ditemukan. Simpan log dan hentikan proses secara tertib jika terblokir.
forum_docs=[]
crawl_log=[]
for source, board in FORUMS.items():
    print("\\n==",source,"==")
    links=discover_thread_links(source, board)
    print("Thread links found:",len(links))
    ok=0
    for url in links:
        doc=extract_thread(source,url)
        if doc:
            forum_docs.append(doc); ok+=1
        crawl_log.append({"source":source,"url":url,"success":bool(doc)})
        time.sleep(DELAY)
    print("Documents extracted:",ok)


\n== MAL ==
Thread links found: 61


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22136\3685115718.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawl_time":datetime.utcnow().isoformat()+"Z"}


Request failed: https://myanimelist.net/forum/?topicid=2286954 ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Request failed: https://myanimelist.net/forum/?topicid=2286956 HTTPSConnectionPool(host='myanimelist.net', port=443): Max retries exceeded with url: /forum/?topicid=2286956 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in vi
Documents extracted: 59
\n== Steam ==
Thread links found: 32
Documents extracted: 32
\n== AniList ==
Thread links found: 5
Documents extracted: 5
\n== PCGamer ==
Thread links found: 0
Documents extracted: 0
\n== AnimeForums ==
Thread links found: 0
Documents extracted: 0
\n== AnimeSuki ==
Thread links found: 32
Access limited: 403 https://forums.animesuki.com/showthread.php?s=1a17e826b04fb371cd71700a9d3b520e&t=113728
Access limited: 403 https://forums.animesuki.com/showthread.php?s=1a17e826b04fb371cd71700a9d3b520e&t=113728&page=8


In [8]:
# Stack Exchange API crawl
def crawl_stackexchange(site, pages=20):
    out=[]
    endpoint="https://api.stackexchange.com/2.3/questions"
    for page in range(1,pages+1):
        params={"order":"desc","sort":"activity","site":site,"pagesize":100,
                "page":page,"filter":"withbody"}
        try:
            r=requests.get(endpoint,params=params,headers=HEADERS,timeout=30)
            if r.status_code in (403,429): print("API rate/access limit",site); break
            r.raise_for_status(); payload=r.json()
            for item in payload.get("items",[]):
                out.append({"source":"StackExchange-"+site,
                    "question_id":item.get("question_id"),"title":item.get("title",""),
                    "content":BeautifulSoup(item.get("body",""),"html.parser").get_text(" "),
                    "url":item.get("link",""),"tags":item.get("tags",[]),
                    "creation_date":item.get("creation_date"),
                    "last_activity_date":item.get("last_activity_date"),
                    "score":item.get("score",0),"answer_count":item.get("answer_count",0)})
            print(site, "page",page,"docs",len(out))
            if not payload.get("has_more"): break
            time.sleep(max(DELAY, payload.get("backoff",0)))
        except Exception as e:
            print("API error",site,page,e); break
    return out

api_docs=[]
for site in API_SITES:
    api_docs.extend(crawl_stackexchange(site))


anime page 1 docs 100
anime page 2 docs 200
anime page 3 docs 300
anime page 4 docs 400
anime page 5 docs 500
anime page 6 docs 600
anime page 7 docs 700
anime page 8 docs 800
anime page 9 docs 900
anime page 10 docs 1000
anime page 11 docs 1100
anime page 12 docs 1200
anime page 13 docs 1300
anime page 14 docs 1400
anime page 15 docs 1500
anime page 16 docs 1600
anime page 17 docs 1700
anime page 18 docs 1800
anime page 19 docs 1900
anime page 20 docs 2000
gaming page 1 docs 100
gaming page 2 docs 200
gaming page 3 docs 300
gaming page 4 docs 400
gaming page 5 docs 500
gaming page 6 docs 600
gaming page 7 docs 700
gaming page 8 docs 800
gaming page 9 docs 900
gaming page 10 docs 1000
gaming page 11 docs 1100
gaming page 12 docs 1200
gaming page 13 docs 1300
gaming page 14 docs 1400
gaming page 15 docs 1500
gaming page 16 docs 1600
gaming page 17 docs 1700
gaming page 18 docs 1800
gaming page 19 docs 1900
gaming page 20 docs 2000


In [ ]:
# Gabungkan, normalisasi kolom, serta simpan korpus mentah beserta log audit
all_docs=forum_docs+api_docs
seen=set(); unique=[]
for d in all_docs:
    key=(d.get("source",""),d.get("url",""))
    if key in seen: continue
    seen.add(key)
    d["content_hash"]=hashlib.sha256((d.get("title","")+" "+d.get("content","")).encode("utf-8")).hexdigest()
    unique.append(d)

raw_path=RAW/"korpus.json"
with raw_path.open("w",encoding="utf-8") as f:
    json.dump(unique,f,ensure_ascii=False,indent=2)

with (AUDIT/"log_crawl.json").open("w",encoding="utf-8") as f:
    json.dump({"source_status":source_status,"thread_log":crawl_log,
               "total_before_unique":len(all_docs),"total_unique":len(unique),
               "crawl_time":datetime.utcnow().isoformat()+"Z"},f,ensure_ascii=False,indent=2)

print("Total documents:",len(unique))
print("Saved:",raw_path)


Total documents: 4096
Saved: D:\semester 5\penelusuran informasi\searchengine\data\raw\korpus.json


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22136\75909327.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawl_time":datetime.utcnow().isoformat()+"Z"},f,ensure_ascii=False,indent=2)


In [10]:
import json
from collections import Counter

with open(RAW / "korpus.json", encoding="utf-8") as f:
    korpus = json.load(f)

print("Total dokumen:", len(korpus))
print("\nDistribusi sumber:")
print(Counter(d["source"] for d in korpus))

print("\nKonten kosong:", sum(not d.get("content") for d in korpus))
print("Judul kosong:", sum(not d.get("title") for d in korpus))
print("URL kosong:", sum(not d.get("url") for d in korpus))
print("Hash unik:", len(set(d.get("content_hash") for d in korpus)))

Total dokumen: 4096

Distribusi sumber:
Counter({'StackExchange-anime': 2000, 'StackExchange-gaming': 2000, 'MAL': 59, 'Steam': 32, 'AniList': 5})

Konten kosong: 0
Judul kosong: 0
URL kosong: 0
Hash unik: 4094
